# RESUME.ipynb — v9 第二場 session 續跑

`model-train_v9.ipynb` 的 160 epochs 約需 15.8 小時（實測 355 s/epoch），
超過 Kaggle 單場上限（互動式 9 小時 / Save & Run All 12 小時），因此第二場用本檔續跑。

### 執行前確認
1. Session 1 已執行 **Step.6** 壓出 `runs.zip` 並上傳成 Kaggle Dataset。
2. 下方 Step.R 的 `RESUME_SRC` 已改成該 Dataset 的實際路徑。

### 本檔的組成
| 段落 | 來源 | 為什麼需要 |
| --- | --- | --- |
| Step.1 | 與主 notebook 逐字相同 | 重建 `/kaggle/working/data.yaml` 與資料集 |
| Step.2 | 與主 notebook 逐字相同 | `bbox_iou` 必須被換成 Wise-Inner-MPDIoU，否則續跑的 loss 與第一場不同 |
| Step.3 | 與主 notebook 逐字相同 | `last.pt` 內含 pickle 過的自訂類別，沒註冊就載不回來 |
| Step.R | 本檔專屬 | 還原 runs 目錄並 `resume=True` |
| Step.6 | 與主 notebook 逐字相同 | 壓出最終輸出 |

**不需要**跑 Step.4（產生 yaml）、Step.4.5（架構驗證）、Step.5（訓練），
架構與超參數都從 `last.pt` 內的紀錄還原。

> ⚠ Step.1 / 2 / 3 是從 `model-train_v9.ipynb` 複製過來的。
> 若日後修改主 notebook 的這幾段，記得同步回本檔，否則續跑的模型會與第一場不一致。

# Step.1 環境檢查與資料集安全準備

In [ ]:
# 1. 檢查 GPU 狀態
!nvidia-smi
# 2. 安裝套件。版本必須釘死：整套注入機制建立在 8.4.121 的
#    parse_model 與 BboxLoss 實作細節上，換版本可能靜默失效（問題 C6）。
#    tensorflow 已移除 —— 交付只需要 PyTorch 權重，那行在 Kaggle 上白花安裝時間。
!pip install -q ultralytics==8.4.121 pyyaml

In [ ]:
import os
import shutil
import sys
import yaml
import ultralytics
print(f"▷ Ultralytics 版本: {ultralytics.__version__}")

In [ ]:
# 1. 定義進度條渲染與資料集複製校驗函式
def render_progress_bar(current, total, task_name="檔案同步複製中", bar_length=25):
    percent = (current / total) * 100 if total > 0 else 100.0
    filled_length = int(bar_length * current // total) if total > 0 else bar_length
    bar = '█' * filled_length + '░' * (bar_length - filled_length)
    
    # 輸出兩行格式：顯示步驟名稱與進度條
    sys.stdout.write(f"\r▷ 正在執行 [{task_name}] | 進度: [{bar}] {percent:5.1f}% ({current}/{total})")
    sys.stdout.flush()
def copy_and_verify_dataset(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        print(f"▷ 錯誤：找不到來源資料集目錄 {src_dir}")
        return False

    # 步驟 1: 收集來源端所有檔案路徑
    src_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            rel_path = os.path.relpath(os.path.join(root, file), src_dir)
            src_files.append(rel_path)
    
    total_files = len(src_files)
    print(f"▷ 來源資料集掃描完成，共計 {total_files} 個檔案")

    # 步驟 2: 逐檔複製並動態刷新進度條
    for idx, rel_path in enumerate(src_files, 1):
        src_path = os.path.join(src_dir, rel_path)
        dst_path = os.path.join(dst_dir, rel_path)
        
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(src_path, dst_path)
        
        # 每 100 筆或最後一筆刷新終端機顯示
        if idx % 100 == 0 or idx == total_files:
            render_progress_bar(idx, total_files, task_name="▷ 檔案同步複製中")
    
    print("\n\n▷ 正在檢查複製檔案")

    # 步驟 3: 複製後檢查機制（比對檔案存在性與 Byte 大小）
    missing_files = []
    corrupted_files = []
    
    dst_files_set = set()
    for root, _, files in os.walk(dst_dir):
        for file in files:
            rel_path = os.path.relpath(os.path.join(root, file), dst_dir)
            dst_files_set.add(rel_path)

    for rel_path in src_files:
        if rel_path not in dst_files_set:
            missing_files.append(rel_path)
        else:
            src_sz = os.path.getsize(os.path.join(src_dir, rel_path))
            dst_sz = os.path.getsize(os.path.join(dst_dir, rel_path))
            if src_sz != dst_sz:
                corrupted_files.append(rel_path)

    # 步驟 4: 輸出校驗結果報告
    print("≡" * 60)
    print("▷ 資料集複製完整性校驗：")
    print(f"  ▶ 來源檔案總數 (Source)     : {len(src_files)}")
    print(f"  ▶ 目標檔案總數 (Destination): {len(dst_files_set)}")
    print(f"  ▶ 遺漏檔案數   (Missing)    : {len(missing_files)}")
    print(f"  ▶ 損毀/大小不符(Corrupted)  : {len(corrupted_files)}")
    
    if not missing_files and not corrupted_files:
        print("▷ 檢查通過")
        print("≡" * 60)
        return True
    else:
        print("▷ 檢查失敗")
        if missing_files:
            print(f"▷ 遺漏檔案: {missing_files[:5]}")
        if corrupted_files:
            print(f"▷ 損毀檔案: {corrupted_files[:5]}")
        print("≡" * 60)
        return False

In [ ]:
# 2. 執行複製與路徑配置
src_dataset_dir = "/kaggle/input/datasets/yentsai9183/datasets-yolo26-v5"
dst_dataset_dir = "/kaggle/working/datasets-yolo26-v5"

# 執行複製與校驗
is_success = copy_and_verify_dataset(src_dataset_dir, dst_dataset_dir)

if is_success:
    # 3. 更新 data.yaml 路徑
    new_yaml_path = "/kaggle/working/data.yaml"
    orig_yaml_path = f"{dst_dataset_dir}/data.yaml"
    
    if os.path.exists(orig_yaml_path):
        with open(orig_yaml_path, 'r', encoding='utf-8') as f:
            yaml_data = yaml.safe_load(f)
        
        yaml_data['path'] = dst_dataset_dir
        yaml_data['train'] = "train/images"
        yaml_data['val'] = "valid/images"
        yaml_data['test'] = "test/images"
        
        with open(new_yaml_path, 'w', encoding='utf-8') as f:
            yaml.safe_dump(yaml_data, f, default_flow_style=False)
        print(f"▷ data.yaml 已更新\n▷ 新設定檔路徑為: {new_yaml_path}")

    # 4. 清理 BOM 標記與舊快取
    print("▷ 正在刪除BOM標籤和Cache")
    modified_count = 0
    deleted_cache_count = 0
    
    for subdir, _, files in os.walk(dst_dataset_dir):
        if "labels" in subdir:
            for file in files:
                if file.endswith('.txt'):
                    file_path = os.path.join(subdir, file)
                    with open(file_path, 'rb') as f:
                        header = f.read(3)
                    if header == b'\xef\xbb\xbf':
                        with open(file_path, 'r', encoding='utf-8-sig') as f:
                            content = f.read()
                        with open(file_path, 'w', encoding='utf-8') as f:
                            f.write(content)
                        modified_count += 1
                        
        for file in files:
            if file.endswith('.cache'):
                os.remove(os.path.join(subdir, file))
                deleted_cache_count += 1
                
    print(f"    ▶ 已自動修正 BOM 檔案數: {modified_count}")
    print(f"    ▶ 已清理舊快取檔案數: {deleted_cache_count}")
    print("▷ Step.1 完成")

# Step.2 加入 Wise-Inner-MPDIoU（定義於 v9_modules.py）

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Step.2 的損失函式與 Step.3 的自訂模組都已移到 v9_modules.py。
#
# 在此之前，同一份 Wise-Inner-MPDIoU 與 StarTripletBlock 同時存在於
# model-train_v9.ipynb、RESUME.ipynb、verify_v9_local.py 三個檔案裡，
# 任何一份漂移都會讓續跑或驗證的模型與訓練時不一致，而且不會報錯（問題 C3）。
# 現在一律從 GitHub 抓同一份，notebook 內不再放第二份定義。
#
# ⚠ Kaggle 的 Notebook 設定裡必須開啟 Internet。
# ══════════════════════════════════════════════════════════════════════════
import subprocess
import sys

URL = ("https://raw.githubusercontent.com/DreamOver9183/"
       "AY2026_Citrus_Pests_and_Diseases_Project/main/Train%20Code/v9/v9_modules.py")
subprocess.run(["curl", "-sSLf", "-o", "/kaggle/working/v9_modules.py", URL], check=True)

sys.path.insert(0, "/kaggle/working")
import v9_modules as v9

import ultralytics
assert v9.REQUIRED_ULTRALYTICS == ultralytics.__version__, \
    f"v9_modules 要求 ultralytics {v9.REQUIRED_ULTRALYTICS}，實際 {ultralytics.__version__}"
print(f"▷ v9_modules {v9.__version__} 已載入")


# ── 安裝 Wise-Inner-MPDIoU ────────────────────────────────────────────
# 定義在 v9_modules.py。相對於最初版本，內含三個數值修正：
#   1. MPD 分母 detach —— 移除「把外接框撐大來降低損失」的退化梯度路徑
#   2. 全程 fp32 計算 —— AMP 之下 c_diag_sq 在特徵圖尺度就會逼近 fp16 上限 65504
#   3. 離群度統計量留在 GPU 且可存取 —— 免去每個 batch 的 .item() 同步，並讓續跑能還原
import torch
import ultralytics.utils.loss

v9.install_loss(ratio=0.7)          # v9 的原始設定；A1b 那條路線用 1.25
assert ultralytics.utils.loss.bbox_iou.__name__ == "bbox_wise_inner_mpdiou"

# 自我檢查：用與 utils/loss.py:133 完全相同的呼叫方式驗證簽名與形狀
_p = torch.tensor([[10.0, 10.0, 30.0, 30.0], [0.0, 0.0, 10.0, 10.0]], requires_grad=True)
_t = torch.tensor([[12.0, 12.0, 32.0, 32.0], [50.0, 50.0, 60.0, 60.0]])
_iou = ultralytics.utils.loss.bbox_iou(_p, _t, xywh=False, CIoU=True)
assert _iou.shape == (2, 1), f"回傳形狀必須為 [N,1]，實際為 {tuple(_iou.shape)}"
assert torch.isfinite(_iou).all(), "回傳值含 NaN/Inf"
(1.0 - _iou).sum().backward()
assert torch.isfinite(_p.grad).all(), "梯度含 NaN/Inf"
print(f"▷ 簽名/形狀/梯度檢查通過  loss(重疊框)={float((1 - _iou[0]).detach()):.4f}  "
      f"loss(不相交框)={float((1 - _iou[1]).detach()):.4f}")
print("▷ Step.2 完成")

# Step.3 註冊 StarTripletBlock（定義於 v9_modules.py）

In [ ]:
# ── 註冊 StarTripletBlock（ADown 沿用 Ultralytics 內建版）──────────────
# 類別定義在 v9_modules.py。註冊會把 StarTripletBlock 綁到 `C2f` 這個既有名稱，
# 因為 parse_model 的 base_modules / repeat_modules 名單寫死，新名稱拿不到
# c1 注入與 width/depth 縮放。yaml 中出現的 C2f 一律代表 StarTripletBlock。
import torch
import ultralytics.nn.tasks

v9.install_modules()
assert ultralytics.nn.tasks.C2f is v9.StarTripletBlock, "C2f 別名未生效"

# 形狀自我檢查（涵蓋 c1==c2 與 c1!=c2 兩種情形）
assert v9.StarTripletBlock(96, 32, n=1)(torch.randn(2, 96, 32, 32)).shape == (2, 32, 32, 32)
assert v9.StarTripletBlock(32, 32, n=2)(torch.randn(2, 32, 32, 32)).shape == (2, 32, 32, 32)
assert ultralytics.nn.tasks.ADown(32, 64)(torch.randn(2, 32, 32, 32)).shape == (2, 64, 16, 16)
print("▷ StarTripletBlock / ADown 形狀檢查通過")
print("▷ 提醒：日後載入 v9 的 best.pt / last.pt 之前，必須先執行本 cell，")
print("        否則 pickle 找不到自訂類別會報 AttributeError。")
print("▷ Step.3 完成")

# Step.R 續跑

`resume=True` 會沿用 `last.pt` 內記錄的全部超參數（`epochs=160`、`batch`、`lr`、`close_mosaic`…），
因此下方不要再傳任何訓練參數。

**不要改用 `time=` 參數限制時數** —— Ultralytics 會用實測 epoch 時間反推並改寫 `args.epochs`
（`engine/trainer.py:619`），續跑的總輪數會被改掉。

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Step.R 續跑
# ═══════════════════════════════════════════════════════════════════════
import os
import re
import shutil
import zipfile
import ultralytics.nn.tasks
import ultralytics.utils.loss
from ultralytics import YOLO

# ── 0. 前置檢查 ───────────────────────────────────────────────────────
# Step.3：last.pt 內含 pickle 過的自訂類別，沒註冊就反序列化不回來
assert ultralytics.nn.tasks.C2f is v9.StarTripletBlock, \
    "C2f 別名未生效 —— 請先執行本檔的 Step.3"
# Step.2：漏跑不會報錯，但會靜默改用內建 CIoU，續跑的 loss 就與第一場不一致
assert ultralytics.utils.loss.bbox_iou.__name__ == "bbox_wise_inner_mpdiou", \
    "bbox_iou 仍是內建 CIoU —— 請先執行本檔的 Step.2，否則續跑的損失函式與第一場不同"
print("▷ 0/4 自訂 loss 與模組均已就位")

# ── 1. 還原上一場 session 的整個 runs 目錄 ────────────────────────────
# ★ 改成你的 Kaggle Dataset 路徑。可指向 Step.6 壓出的 runs.zip，或解開後的 runs 資料夾。
RESUME_SRC = "/kaggle/input/v9-session1/runs.zip"

RUN_NAME = "YOLO26n_P2_Citrus_MuSGD_v9"
WORK_RUNS = "/kaggle/working/runs"
RUN_DIR = f"{WORK_RUNS}/detect/{RUN_NAME}"
WDIR = f"{RUN_DIR}/weights"

assert os.path.exists(RESUME_SRC), f"找不到 {RESUME_SRC}，請確認 Kaggle Dataset 路徑"
os.makedirs(WORK_RUNS, exist_ok=True)
if RESUME_SRC.endswith(".zip"):
    with zipfile.ZipFile(RESUME_SRC) as z:
        z.extractall(WORK_RUNS)
else:
    shutil.copytree(RESUME_SRC, WORK_RUNS, dirs_exist_ok=True)
assert os.path.isdir(WDIR), (
    f"還原後找不到 {WDIR}。Step.6 的 runs.zip 內層應為 detect/{RUN_NAME}/weights/...，"
    "若結構不同請調整 RESUME_SRC 或解壓目標。")
print(f"▷ 1/4 已還原 runs 目錄：{sorted(os.listdir(WDIR))}")

# ── 2. 挑出可續跑的 checkpoint ────────────────────────────────────────
# final_eval() 會對 last.pt / best.pt 執行 strip_optimizer()，把 epoch 改成 -1
# 並清掉 optimizer/EMA，那種檔案無法續跑。可用的優先序：
#   resume_from.pt（Session 1 的 callback 在 strip 之前留的複本）
#   epochN.pt（save_period 產出，不會被 strip，取 N 最大者）
#   last.pt（只有在 Session 1 是被強制中斷、來不及 strip 時才可用）
import torch

def usable(path):
    """回傳該 ckpt 已完成的輪數；不可續跑則回傳 None。"""
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"    {os.path.basename(path)}: 讀取失敗 {type(e).__name__}")
        return None
    ep, opt = ck.get("epoch", -1), ck.get("optimizer")
    del ck
    return ep + 1 if (ep is not None and ep >= 0 and opt is not None) else None

cands = [os.path.join(WDIR, "resume_from.pt")]
cands += sorted((os.path.join(WDIR, f) for f in os.listdir(WDIR)
                 if re.fullmatch(r"epoch\d+\.pt", f)),
                key=lambda p: int(re.search(r"\d+", os.path.basename(p)).group()), reverse=True)
cands.append(os.path.join(WDIR, "last.pt"))

CKPT, DONE = None, None
for c in cands:
    if not os.path.exists(c):
        continue
    n = usable(c)
    print(f"    {os.path.basename(c):<18} " + (f"已完成 {n} 輪，可續跑" if n else "已被 strip，不可續跑"))
    if n and CKPT is None:
        CKPT, DONE = c, n

assert CKPT, (
    "沒有任何可續跑的 checkpoint。所有權重的 epoch 都是 -1，代表 Session 1 的訓練迴圈"
    "正常結束並執行了 strip_optimizer()——若那是因為 patience=30 觸發早停，"
    "表示訓練本來就已收斂完成，不需要續跑。")
print(f"▷ 2/4 選用 {os.path.basename(CKPT)}（已完成 {DONE} 輪）")

# ── 2.5 還原自訂 loss 的離群度統計量 ──────────────────────────────────
# WIoU v3 的 iou_mean 是滑動平均，不在 checkpoint 裡。少了這步，第二場
# session 會從 1.0 重新暖機，focus 係數與第一場對不上（問題 B4）。
_ws = os.path.join(WDIR, "wiou_state.txt")
if os.path.exists(_ws):
    v9.set_wiou_state(float(open(_ws).read().strip()))
    print(f"▷ 已還原 WIoU 離群度統計量：{v9.wiou_state():.4f}")
else:
    print("▷ 找不到 wiou_state.txt，離群度統計量將從 1.0 重新暖機")
    print("  （影響有限，但第一場的 focus 係數無法完全重現）")


# ── 3. 續跑 ───────────────────────────────────────────────────────────
# resume=True 會沿用 ckpt 內記錄的全部超參數（epochs=160、batch、lr、close_mosaic…），
# 因此下方不要再傳任何訓練參數。
model = YOLO(CKPT)
print(f"▷ 3/4 開始續跑：第 {DONE + 1} 輪 → 第 160 輪")
results = model.train(resume=True)
print("▷ 4/4 RESUME 訓練完畢")

# Step.6 Output整理

In [ ]:
import os
import shutil

# 定義工作區與 uns目錄路徑
working_dir = "/kaggle/working"
runs_dir = os.path.join(working_dir, "runs")
zip_output_path = os.path.join(working_dir, "runs")

# 檢查runs目錄並壓縮
if os.path.exists(runs_dir):
    print("▷ 正在壓縮訓練輸出")
    shutil.make_archive(zip_output_path, 'zip', runs_dir)
    
    zip_full_path = f"{zip_output_path}.zip"
    if os.path.exists(zip_full_path):
        size_mb = os.path.getsize(zip_full_path) / (1024 * 1024)
        print(f"▷ 壓縮成功 {zip_full_path} ({size_mb:.2f} MB)")
else:
    print(f"▷ 壓縮失敗: 找不到 '{runs_dir}' 資料夾")